This notebook is designed to create a new simulation based on an existing simulation. That we you ensure everything is the same except something in particular.

At first, it is being used to keep all synapses and original background inputs the same while turning clustered inputs into background inputs

Define the simulation you want to base the new one off of

In [ ]:
# original_sim_dir = '/home/drfrbc/Neural-Modeling/simulations/2025-06-09-12-20-tuft_exc_clusters_30sec_sta/complex'
original_sim_dir = '/home/drfrbc/Neural-Modeling/simulations/2025-06-12-13-40-pink_and_delay_no_clustering/complex'

new_sim_dir = '/home/drfrbc/Neural-Modeling/simulations/2025-06-12-14-11-pink_and_delay_no_clustering_direct_remake/complex'



In [2]:
import shutil
import os

import sys
sys.path.append('../')
sys.path.append('../Modules/')


In [3]:
os.chdir('../simulations')

copy the simulation to the new one

In [4]:
# create the new simulation directory if it doesn't exist
if not os.path.exists(new_sim_dir):
    os.makedirs(new_sim_dir)

parameters_file = f'{new_sim_dir}/parameters.pickle'
synapses_file = f'{new_sim_dir}/synapses.csv'
segments_file = f'{new_sim_dir}/segment_data.csv'

# copy over the parameters and synapses to the new simulation directory
shutil.copyfile(f'{original_sim_dir}/parameters.pickle', parameters_file)
shutil.copyfile(f'{original_sim_dir}/synapses.csv', synapses_file)
shutil.copyfile(f'{original_sim_dir}/segment_data.csv', segments_file)

'/home/drfrbc/Neural-Modeling/simulations/2025-06-11-11-21-no_clustering_30sec_sta_inc_tuft_exc_dens_more/complex/segment_data.csv'

load data

In [5]:
from Modules import analysis
import pandas as pd

# load parameters from the new simulation directory
parameters = analysis.DataReader.load_parameters(new_sim_dir)

# load synapses from the new simulation directory
synapses = pd.read_csv(synapses_file)

--No graphics will be displayed.


### Make your changes to the data

In [6]:
synapses.columns

Index(['name', 'modfile', 'P_0', 'initW', 'cell2cell_type', 'seg_id',
       'gbar_ampa', 'gbar_nmda', 'gbar_gaba', 'spike_train',
       'pc_mean_firing_rate', 'functional_group', 'presynaptic_cell'],
      dtype='object')

In [ ]:
# change clustered synapses to background spike trains

import numpy as np
import pandas as pd
from functools import partial
from Modules.spike_generator import PoissonTrainGenerator
import concurrent.futures

def generate_fg_trace(spike_train_mode, props, h_tstop, fg_id=None, fg=None):
    """
    Returns the modulatory trace (lambda over time) for a functional group.
    Respects FG 'modulation_mode' override if present.
    """
    mode = spike_train_mode if fg is None else fg.get('modulation_mode', spike_train_mode)
    if mode == 'standard':
        return np.ones(h_tstop)
    elif mode == 'pink_noise':
        fg_trace = PoissonTrainGenerator.generate_lambdas_from_pink_noise(
            num=h_tstop, random_state=random_state)
        mean_val = np.mean(fg_trace)
        if mean_val == 0:
            raise ValueError("Mean value of pink noise trace is zero; cannot normalize.")
        return fg_trace / mean_val
    elif mode == 'rhythmic':
        base = np.ones(h_tstop)
        freq = props.get('rhythmic_frequency', None)
        depth = props.get('rhythmic_depth', None)
        delta_t = getattr(parameters, 'delta_t', 1)
        if freq is None or depth is None:
            raise ValueError("Both 'rhythmic_frequency' and 'rhythmic_depth' must be set for rhythmic mode.")
        return PoissonTrainGenerator.rhythmic_modulation(base, freq, depth, delta_t)
    # elif mode == 'delay':
    #     shift = delay_config.get('delay_shift', None)
    #     if shift is None:
    #         raise ValueError("delay_shift must be specified in delay_config for 'delay' mode.")
    #     ref_synapse_type = delay_config.get('ref_synapse_type', 'exc')
    #     ref_sec_type = delay_config.get('ref_sec_type', sec_type)
    #     ref_fg_id = delay_config.get('ref_fg_id', fg_id)
    #     ref_pc_id = delay_config.get('ref_pc_id', None)
    #     trains = collect_reference_spike_trains(
    #         ref_synapse_type, ref_sec_type, ref_fg_id, ref_pc_id)
    #     if not all(isinstance(train, (np.ndarray, list)) and len(train) > 0 for train in trains):
    #         raise ValueError("All reference spike trains for delay must be non-empty arrays/lists.")
    #     delayed_lambdas = PoissonTrainGenerator.generate_lambdas_by_delaying(h_tstop, trains)
    #     return delayed_lambdas
    else:
        raise NotImplementedError(f"Unrecognized spike_train_mode: {mode}")

def process_row(args):
    idx, row, parameters, tstop, random_state = args
    # Synapse type
    if 'exc' in row['name']:
        prop_dict = parameters.exc_syn_properties
    elif 'inh' in row['name']:
        prop_dict = parameters.inh_syn_properties
    else:
        raise ValueError(f"Unknown synapse type in name: {row['name']}")
    sec_type = next((key for key in prop_dict.keys() if key in row['name']), None)
    if sec_type is None:
        raise ValueError(f"Cannot determine sec_type for {row['name']}")
    mean_fr_dist = prop_dict[sec_type]['mean_firing_rate_distribution']
    mean_fr_sampler = partial(mean_fr_dist['function'], **mean_fr_dist['params'], size=1)
    spike_train_mode = prop_dict[sec_type]['spike_train_mode'] # generate background for this type
    background_firing_rate_timecourse = np.ones(tstop)
    pc_mean_firing_rate = mean_fr_sampler(size=1)
    pc_firing_rate_timecourse = PoissonTrainGenerator.shift_mean_of_lambdas(
        lambdas=background_firing_rate_timecourse,
        desired_mean=pc_mean_firing_rate
    )
    spike_train = PoissonTrainGenerator.generate_spike_train(
        lambdas=pc_firing_rate_timecourse,
        random_state=random_state
    )
    return idx, np.array(spike_train.spike_times), pc_mean_firing_rate

def assign_background_spike_trains_parallel(
    synapses: pd.DataFrame,
    parameters,
    tstop: int,
    random_state: np.random.RandomState
):
    new_synapses = synapses.copy()
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(executor.map(
            process_row,
            [(idx, row, parameters, tstop, random_state) for idx, row in new_synapses.iterrows()]
        ))
    for idx, spike_train, pc_mean_firing_rate in results:
        new_synapses.at[idx, 'spike_train'] = spike_train
        new_synapses.at[idx, 'pc_mean_firing_rate'] = pc_mean_firing_rate
    return new_synapses


import numpy as np

# Save a copy of your original DataFrame
synapses_original = synapses.copy()


# Assuming synapses is loaded and parameters is available
random_state = np.random.RandomState(42)  # or whatever seed you want

# Filter for clustered synapses
clustered_mask = ((synapses['functional_group'] != -1.0) | (synapses['presynaptic_cell'] != -1.0))
clustered_synapses = synapses[clustered_mask].copy()

# # Check example spike trains before
# print("Old spike trains for clustered synapses:")
# print(synapses.loc[clustered_synapses.index, 'spike_train'].head())

# Reassign background spike trains
tstop = parameters.h_tstop
backgrounded = assign_background_spike_trains_parallel(clustered_synapses, parameters, tstop, random_state)

# Save or overwrite in the original DataFrame if needed:
synapses.loc[clustered_synapses.index, ['spike_train', 'pc_mean_firing_rate']] = \
    backgrounded[['spike_train', 'pc_mean_firing_rate']]

# Save back to disk if needed
synapses.to_csv(os.path.join(new_sim_dir, "synapses.csv"), index=False)

# verify the changes
# Check after
# Print BEFORE and AFTER for the SAME indices!
for idx in clustered_synapses.index[:5]:
    print(f"Index {idx}:")
    print("  Original:", synapses_original.at[idx, 'spike_train'])
    print("  Current:", synapses.at[idx, 'spike_train'])



Index 0:
  Original: [  302   345   479   610   636   687   762   775   822  1124  1132  1315
  1345  1515  1593  1661  1856  1909  1911  1948  2032  2133  2315  2430
  2658  2694  2697  2760  2770  2961  3007  3179  3273  3328  3437  3518
  3686  3835  3992  4238  4641  4815  4952  5195  5235  5332  5675  5738
  5838  5860  5950  6029  6072  6088  6192  6328  6391  6397  6491  6724
  7000  7018  7070  7077  7204  7390  7417  7598  7630  7659  7694  7809
  7901  7961  8027  8072  8082  8184  8222  8379  8489  8825  8855  9753
  9761  9774 10350 10376 10474 10603 10642 10734 10777 11152 11355 11542
 11622 11674 11890 11930 12321 12512 12536 12627 12632 12638 12675 12935
 12941 12955 13121 13341 13346 13701 13887 14354 14535 14878 14961 15031
 15097 15183 15330 15771 15823 15825 15949 15970 15979 16111 16223 16872
 16923 17163 17185 17335 17614 17660 17726 17973 18282 18346 18371 18489
 18678 19099 19291 19408 19529 19567 19605 19831 19853 19972 20119 20316
 20506 20549 20551 20584 20718

Create a new simulation with the same synapse firing rates, but no clustering.

In [8]:
# import numpy as np
# import pandas as pd
# from Modules.spike_generator import PoissonTrainGenerator
# import concurrent.futures
# import os

# def process_row_from_existing_rate(args):
#     idx, row, parameters, tstop, random_state = args
#     # Synapse type
#     if 'exc' in row['name']:
#         prop_dict = parameters.exc_syn_properties
#     elif 'inh' in row['name']:
#         prop_dict = parameters.inh_syn_properties
#     else:
#         raise ValueError(f"Unknown synapse type in name: {row['name']}")

#     sec_type = next((key for key in prop_dict.keys() if key in row['name']), None)
#     if sec_type is None:
#         raise ValueError(f"Cannot determine sec_type for {row['name']}")

#     # Use the mean firing rate from the DataFrame directly
#     pc_mean_firing_rate = row['pc_mean_firing_rate']

#     # Create a firing rate timecourse: constant in this case
#     pc_firing_rate_timecourse = np.ones(tstop) * pc_mean_firing_rate

#     # Optionally, if you want non-integer time steps, adjust accordingly
#     spike_train = PoissonTrainGenerator.generate_spike_train(
#         lambdas=pc_firing_rate_timecourse,
#         random_state=random_state
#     )
#     return idx, np.array(spike_train.spike_times), pc_mean_firing_rate

# def assign_background_spike_trains_with_existing_rates_parallel(
#     synapses: pd.DataFrame,
#     parameters,
#     tstop: int,
#     random_state: np.random.RandomState
# ):
#     new_synapses = synapses.copy()
#     with concurrent.futures.ThreadPoolExecutor() as executor:
#         results = list(executor.map(
#             process_row_from_existing_rate,
#             [(idx, row, parameters, tstop, random_state) for idx, row in new_synapses.iterrows()]
#         ))
#     for idx, spike_train, pc_mean_firing_rate in results:
#         new_synapses.at[idx, 'spike_train'] = spike_train
#         new_synapses.at[idx, 'pc_mean_firing_rate'] = pc_mean_firing_rate
#     return new_synapses

# # Example usage:
# # Save a copy of your original DataFrame
# synapses_original = synapses.copy()

# random_state = np.random.RandomState(42)  # Or use a passed-in value

# # Filter for clustered synapses
# clustered_mask = ((synapses['functional_group'] != -1.0) | (synapses['presynaptic_cell'] != -1.0))
# clustered_synapses = synapses[clustered_mask].copy()

# tstop = parameters.h_tstop
# backgrounded = assign_background_spike_trains_with_existing_rates_parallel(clustered_synapses, parameters, tstop, random_state)

# # Overwrite only the updated fields in the main DataFrame
# synapses.loc[clustered_synapses.index, ['spike_train', 'pc_mean_firing_rate']] = \
#     backgrounded[['spike_train', 'pc_mean_firing_rate']]

# # Save back to disk if needed
# synapses.to_csv(os.path.join(new_sim_dir, "synapses.csv"), index=False)

# # Print BEFORE and AFTER for a few indices
# for idx in clustered_synapses.index[:5]:
#     print(f"Index {idx}:")
#     print("  Original:", synapses_original.at[idx, 'spike_train'])
#     print("  Current:", synapses.at[idx, 'spike_train'])
